# Linear Regression Implementation


In [1]:
import numpy as np
import pandas as pd
import math

In [2]:
#Constants 

FEATURES = [
    'year',
    'population',
    'gdp',
    'coal_consumption',
    'oil_consumption',
    'gas_consumption',
    'nuclear_consumption',
    'hydro_consumption',
    'solar_consumption',
    'wind_consumption',
    'biofuel_consumption'
]

TARGET = 'primary_energy_consumption'

In [3]:
#Methods for Lienar Regressions for Calculations
def pre_process(df):
    #data = df[FEATURES + [TARGET]].copy()
    data = df
    data = data.dropna()
    # print(data.head())
    # print(data.shape)
    return data

def rmse(actual_Y, predicted_Y):
    summ = actual_Y - predicted_Y
    rmse = np.sqrt(np.mean((summ)**2))
    return rmse

def smape(actual_Y, predicted_Y):
    num = np.abs(actual_Y - predicted_Y)
    den = (np.abs(actual_Y) + np.abs(predicted_Y)) / 2
    return np.mean(num/den) * 100

def z_score(train_data, validate_data):

    mean = np.mean(train_data, axis=0)
    std_dev = np.std(train_data, axis=0)

    std_dev[std_dev == 0] = 1

    z_train_score = (train_data - mean) /std_dev
    z_validate_score = (validate_data - mean) /std_dev

    return z_train_score, z_validate_score


In [4]:
# Read the FILE
path = "../data/df_model.csv"

df = pd.read_csv(path)

clean_df = pre_process(df)

length = len(clean_df)

In [5]:
#Shuffled Data
shuffled_df = clean_df.sample(frac=1, random_state=42).reset_index(drop=True)

X = shuffled_df.drop(TARGET, axis=1).values
Y = shuffled_df[TARGET].values

In [6]:
#Split the data
split_data = math.ceil(2/3 * length)

#2/3
X_train_data = X[:split_data]
Y_train_data = Y[:split_data]

#1/3
X_validate_data = X[split_data:]
Y_validate_data = Y[split_data:]


In [7]:
#ZSCORE
X_train_data_Z, X_validate_data_Z = z_score(X_train_data, X_validate_data)


In [8]:
#Bias
X_bias_train_data = np.hstack([np.ones((X_train_data_Z.shape[0], 1)), X_train_data_Z])
X_bias_validate_data = np.hstack([np.ones((X_validate_data_Z.shape[0], 1)), X_validate_data_Z])

In [9]:
#Weights
XtX = X_bias_train_data.T @ X_bias_train_data
XtY = X_bias_train_data.T @ Y_train_data

weight = np.linalg.pinv(XtX) @ XtY


In [10]:
#Predictions
Y_train_pred = X_bias_train_data @ weight
Y_validate_pred = X_bias_validate_data @ weight

training_data_rmse = rmse(Y_train_data, Y_train_pred)
validate_data_rmse = rmse(Y_validate_data, Y_validate_pred)

training_data_smape = smape(Y_train_data, Y_train_pred)
validate_data_smape = smape(Y_validate_data, Y_validate_pred)

In [11]:
#Output
print(f"Training Data RMSE: {(training_data_rmse)} => {(training_data_rmse):.2f}")
print(f"Validate Data RMSE: {(validate_data_rmse)} => {(validate_data_rmse):.2f}")

print(f"Training Data SMAPE: {(training_data_smape)} => {(training_data_smape):.2f}%")
print(f"Validate Data SMAPE: {(validate_data_smape)} => {(validate_data_smape):.2f}%")


Training Data RMSE: 114.30117161890206 => 114.30
Validate Data RMSE: 129.5374453493723 => 129.54
Training Data SMAPE: 7.663949144568159 => 7.66%
Validate Data SMAPE: 7.874091780516943 => 7.87%


In [ ]:
results_linearReg = pd.DataFrame({
    'Model': ['Linear Regression', 'Linear Regression'],
    'Dataset': ['Train', 'Test'],
    'RMSE': [training_data_rmse, validate_data_rmse],
    'sMAPE': [training_data_smape, validate_data_smape]
})